In [0]:
caminho_silver = "/Volumes/pratica/pratica_dbs/bronze/anac_raw_bronze/"

df_anac_silver = spark.read.format("parquet").load(caminho_silver)
display(df_anac_silver.limit(10))

In [0]:
# 1 passo padronização de nomes de colunas usando o comando withColumnsRenamed({})
# 2 passo alterar todas as colunas que vieream como string, exemplo: numero_de_Assentos e Lesoes_Fatais_Passageiros.
# 3 Tratar valores nulos nas colunas que foram manipuladas, preenche-los com o número 0.
# O uso do cast() me retornou um erro de [CAST_INVALID_INPUT], como o Databricks opera com ANSI Mode ativado por padrão esse erro ocorreu.
# Esse erro ocorreo por que tentei converter um falso "NULL"(string) para INT. A solucao foi usar, expr("try_cast(...)") para forçar a conversão de dados inválidos em NULL real,
# permitindo a higienização posterior com .na.fill(0) sem quebrar o pipeline.

from pyspark.sql.functions import expr, col, when

# mudando o nome de algumas colunas, para entender como withcolumnsRenamed funciona.
new_name_columns = {
    "Numero_da_Ocorrencia": "numero_da_ocorrencia",
    "Numero_da_Ficha": "numero_da_ficha",
    "Operador_Padronizado": "operador_padronizado",
    "Classificacao_da_Ocorrencia": "classificacao_da_ocorrencia",
    "Data_da_Ocorrencia": "data_da_ocorrencia",
}

df_anac_silver_cleaning = (
    df_anac_silver.withColumnsRenamed(new_name_columns)
    .withColumn(
        "lesoes_fatais_passageiros", expr("try_cast(Lesoes_Fatais_Passageiros AS INT)")
    )
    .withColumn("Numero_de_Assentos", expr("try_cast(Numero_de_Assentos AS INT)"))
    .withColumn(
        "Operador_Padronizado",
        when(col("Operador_Padronizado") == "null", "NAO_IDENTIFICADO").otherwise(
            col("Operador_Padronizado")
        ),
    )
    .na.fill(
        {
            "lesoes_fatais_passageiros": 0,
            "Numero_de_Assentos": 0,
        }
    )  # tratamento de nulos colocando valores 0.
    .dropDuplicates(["numero_da_ocorrencia"])
)

display(df_anac_silver_cleaning.limit(10))

In [0]:
df_anac_silver_cleaning.printSchema()

In [0]:
##  O save foi feito em Parquet com compressao Snappy, para melhor velocidade e leitura.

df_anac_silver_path = "/Volumes/pratica/pratica_dbs/silver_anac/anac_cleaned"

(
    df_anac_silver_cleaning.write.format("parquet")
    .options(compression="Snappy")
    .mode("overwrite")
    .save(df_anac_silver_path)
)

In [0]:
display(dbutils.fs.ls("/Volumes/pratica/pratica_dbs/silver_anac/anac_cleaned"))